# FINAL SUBMISSION NOTEBOOK (FULL + FIXED)

In [1]:

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2TokenizerFast

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

VOCAB_SIZE=50257
N_EMBD=512
N_HEAD=8
N_LAYER=8
BLOCK_SIZE=256
DROPOUT=0.1


/home/usera19/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:

class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)).bool())
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(~self.tril[:T, :T], float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.attn_dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)
        ])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.resid_dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa  = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff  = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout, device):
        super().__init__()
        self.device = device
        self.block_size = block_size

        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])

        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding_table.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)
        pos = self.position_embedding_table(torch.arange(T, device=self.device))
        x   = self.blocks(tok + pos)
        x   = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, None


In [3]:

@torch.no_grad()
def generate(model, idx, max_new_tokens=100,
             temperature=0.6,
             top_k=20,
             top_p=0.85,
             repetition_penalty=1.2,
             eos_token_id=50256):

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits,_ = model(idx_cond)
        logits = logits[:, -1, :]

        for token_id in set(idx[0].tolist()):
            if logits[0, token_id] > 0:
                logits[0, token_id] /= repetition_penalty
            else:
                logits[0, token_id] *= repetition_penalty

        logits = logits / temperature

        if top_k:
            v,_ = torch.topk(logits, top_k)
            logits[logits < v[:, -1, None]] = -float("inf")

        probs = torch.softmax(logits, dim=-1)
        probs = torch.clamp(probs, min=1e-8)
        probs = probs / probs.sum(dim=-1, keepdim=True)

        if top_p < 1.0:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cumulative = torch.cumsum(sorted_probs, dim=-1)

            mask = cumulative > top_p
            mask[...,1:] = mask[...,:-1].clone()
            mask[...,0] = False

            sorted_probs[mask] = 0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

            next_token = torch.multinomial(sorted_probs,1)
            next_token = sorted_idx.gather(-1,next_token)
        else:
            next_token = torch.multinomial(probs,1)

        idx = torch.cat([idx,next_token],dim=1)

        if next_token[0,0].item() == eos_token_id:
            break

    return idx


In [4]:

checkpoint_paths = [
    ("WikiText","WikiText.pt"),
    ("BookCorpus_v1","BookCorpusv1.pt"),
    ("BookCorpus_v2","BookCorpusv2.pt"),
]

models=[]

for name,path in checkpoint_paths:
    print("Loading", name)
    m = GPT(VOCAB_SIZE,N_EMBD,N_HEAD,N_LAYER,BLOCK_SIZE,DROPOUT,DEVICE).to(DEVICE)

    raw_state = torch.load(path, map_location=DEVICE)
    if isinstance(raw_state, dict) and "model_state_dict" in raw_state:
        raw_state = raw_state["model_state_dict"]

    cleaned = {k.replace("_orig_mod.",""): v for k,v in raw_state.items()}

    m.load_state_dict(cleaned, strict=True)
    m.eval()

    models.append((name,m))

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
print("Models loaded")


Loading WikiText
Loading BookCorpus_v1
Loading BookCorpus_v2
Models loaded


In [5]:

prompts = ["The history of artificial intelligence began in",
"The theory of relativity states that",
"In the early 20th century, scientists discovered",
"The main purpose of government is to",
"Climate change is primarily caused by",
"The capital of France is",
"Water boils at a temperature of",
"The largest planet in the solar system is",
"A computer consists of",
"The process of photosynthesis involves",
"Once upon a time, there was a small village",
"The story begins with a young boy who",
"In a distant future, humans have developed",
"She opened the door slowly and saw",
"The night was dark and full of",
"If a person walks 5 km every day, then",
"When supply increases and demand decreases,",
"If two objects collide in space,",
"The main difference between speed and velocity is",
"Energy cannot be created or destroyed because",
"He looked at her and said",
"This idea can be interpreted as",
"The results of the experiment were",
"One possible explanation is that",
"In conclusion, the findings suggest",
"asdfghjkl qwerty",
"The quantum banana exploded into",
"12345 times the value of",
"He go to store yesterday and",
"Random words without meaning because"]


In [6]:

def generate_text(model, prompt):
    ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    out = generate(model, ids)
    return tokenizer.decode(out[0], skip_special_tokens=True)


In [7]:

print("=== GENERATION ===")
for p in prompts:
    print("\nPROMPT:", p)
    for name,m in models:
        print(f"\n{name}:")
        print(generate_text(m,p))


=== GENERATION ===

PROMPT: The history of artificial intelligence began in

WikiText:
The history of artificial intelligence began in the early 1960s .


BookCorpus_v1:
The history of artificial intelligence began in the late 1980s , with the introduction of a new series of articles that were published by the United States on April 1 , 1991 . The first edition was released on March 29 , 1992 and contained two books : one book titled " Lost Earth " ( PDF ) , an anthology of essays from the American Association of America 's journal Star Wars ( ISBN 978 - 0 - 01 - 2 ) , and another book titled " Lost Earth " ( ISBN 0 - 921 - 3 ) .
In 1994

BookCorpus_v2:
The history of artificial intelligence began in the early 20th century . The first recorded discovery of a human - based object was discovered by the American astronomer John L. C. , who described it as " an alien - based device that can be easily manipulated by its own kind ... It is possible to find out how much information you have a

In [8]:

def compute_perplexity(model,text):
    ids = tokenizer.encode(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits,_ = model(ids)
        shift_logits = logits[:,:-1,:]
        shift_labels = ids[:,1:]
        loss = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1)
        )
    return torch.exp(loss).item()

eval_texts = [
"The history of science has always been influenced by",
"In the modern world, technology plays a crucial role in",
"Once upon a time, there lived a king who",
"The experiment results indicated that",
"Human behavior is often shaped by"
]

print("\n=== PERPLEXITY ===")
for name,m in models:
    scores=[compute_perplexity(m,t) for t in eval_texts]
    print(name, sum(scores)/len(scores))



=== PERPLEXITY ===
WikiText 310.6287380218506
BookCorpus_v1 167.9368019104004
BookCorpus_v2 212.0330810546875


In [9]:

def repetition_score(text):
    w=text.split()
    return 1-(len(set(w))/len(w))

def diversity_score(text):
    w=text.split()
    return len(set(w))/len(w)

print("\n=== METRICS ===")
for name,m in models:
    print(name)
    for p in prompts[:5]:
        out = generate_text(m,p)
        print("Rep:", round(repetition_score(out),2),
              "Div:", round(diversity_score(out),2))



=== METRICS ===
WikiText
Rep: 0.18 Div: 0.82
Rep: 0.3 Div: 0.7
Rep: 0.27 Div: 0.73
Rep: 0.16 Div: 0.84
Rep: 0.06 Div: 0.94
BookCorpus_v1
Rep: 0.18 Div: 0.82
Rep: 0.3 Div: 0.7
Rep: 0.29 Div: 0.71
Rep: 0.25 Div: 0.75
Rep: 0.17 Div: 0.83
BookCorpus_v2
Rep: 0.22 Div: 0.78
Rep: 0.38 Div: 0.62
Rep: 0.19 Div: 0.81
Rep: 0.29 Div: 0.71
Rep: 0.21 Div: 0.79
